# Día 4: Regresión Logística y Clasificación Binaria

¡Bienvenidos a la Clase 4! Hoy daremos un paso crucial. Pasaremos de predecir valores continuos (como el precio de una casa o las ventas del próximo mes) a **clasificar categorías**. 

En los negocios, muchas preguntas clave tienen respuestas binarias (Sí/No):
- ¿El cliente cancelará su suscripción? (Sí = 1, No = 0)
- ¿Este correo es Spam? (Sí = 1, No = 0)
- ¿El usuario comprará el producto tras ver el anuncio? (Sí = 1, No = 0)

Para esto, utilizaremos la **Regresión Logística**. A pesar de tener la palabra "Regresión" en su nombre, es un algoritmo de clasificación.

## 1. El Problema de la Regresión Lineal para Clasificación

Imagina que queremos predecir si un estudiante aprobará (1) o reprobará (0) un examen basándonos en las horas de estudio.

Veamos qué pasa si intentamos usar una línea recta (Regresión Lineal) para este problema.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Configuramos el estilo de los gráficos
sns.set_theme(style="whitegrid")

# Datos simulados: Horas de estudio vs Aprobado(1)/Reprobado(0)
horas = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]).reshape(-1, 1)
aprobado = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

# Entrenamos una Regresión Lineal
modelo_lineal = LinearRegression()
modelo_lineal.fit(horas, aprobado)

# Generamos predicciones para la línea
horas_test = np.linspace(0, 7, 100).reshape(-1, 1)
predicciones_lineales = modelo_lineal.predict(horas_test)

# Graficamos
plt.figure(figsize=(8, 5))
plt.scatter(horas, aprobado, color='red', label='Datos Reales (0=Reprueba, 1=Aprueba)')
plt.plot(horas_test, predicciones_lineales, color='blue', label='Regresión Lineal')
plt.axhline(1, color='gray', linestyle='--')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Horas de Estudio')
plt.ylabel('Probabilidad de Aprobar')
plt.title('¿Por qué falla la Regresión Lineal en Clasificación?')
plt.legend()
plt.show()

### Análisis:
Observa la línea azul. ¡Predice probabilidades menores a 0 (negativas) para pocas horas de estudio, y mayores a 1 (más del 100%) para muchas horas!

Esto no tiene sentido. Necesitamos una función que "aplaste" la línea para que siempre viva entre 0 y 1. Esa es la **Función Sigmoide** (o logística).

## 2. La Función Sigmoide y Caso de Negocio: Social Network Ads

La Regresión Logística utiliza la función Sigmoide: $P(y=1) = \frac{1}{1 + e^{-z}}$ donde $z$ es la ecuación de una línea recta.

**Analogía del Tiro al Blanco:** Piensa en la función sigmoide como la "nota final" del algoritmo. El algoritmo lanza un dardo (calcula un valor con una fórmula lineal). Si el dardo cae muy a la izquierda, la nota es casi 0 (no comprará). Si cae muy a la derecha, la nota es casi 1 (sí comprará). Si cae en el medio, la nota es 0.5 (está dudoso).

Para decidir finalmente, usamos una regla (Umbral o *Threshold*): **Si la nota es $\geq 0.5$, predecimos 1. Si no, predecimos 0.**

### Caso de Negocio
Tenemos datos de usuarios de una red social: su Edad y su Salario Estimado anual. Queremos predecir si comprarán (1) o no comprarán (0) un nuevo automóvil SUV anunciado en la plataforma.

In [ ]:
from sklearn.datasets import make_classification

# Generamos un dataset simulado de 'Social Network Ads' para el ejercicio
X, y = make_classification(
    n_samples=400, n_features=2, n_redundant=0, n_informative=2,
    random_state=42, n_clusters_per_class=1, flip_y=0.1, class_sep=1.2
)

# Transformamos los datos a una escala realista de Edad (18-60) y Salario (20k - 150k)
X[:, 0] = np.interp(X[:, 0], (X[:, 0].min(), X[:, 0].max()), (18, 60))
X[:, 1] = np.interp(X[:, 1], (X[:, 1].min(), X[:, 1].max()), (20000, 150000))

df = pd.DataFrame(X, columns=['Edad', 'Salario_Estimado'])
df['Compro'] = y

display(df.head())

# Visualizamos los datos
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Edad', y='Salario_Estimado', hue='Compro', data=df, palette='Set1', alpha=0.8)
plt.title('Edad vs Salario Estimado (Compradores de SUV)')
plt.show()

### Entrenamiento del Modelo de Regresión Logística

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Separamos X (variables predictoras) e y (variable objetivo)
X_features = df[['Edad', 'Salario_Estimado']]
y_target = df['Compro']

# 2. Dividimos en Entrenamiento (70%) y Prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size=0.3, random_state=42)

# 3. IMPORTANTE: Escalamos los datos (Estandarización)
# La regresión logística es sensible a la escala de las variables (Salario vs Edad)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Instanciamos y Entrenamos el Modelo
modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_train_scaled, y_train)

print("¡Modelo entrenado exitosamente!")

## 3. La Frontera de Decisión (Decision Boundary)

¿Cómo separa el algoritmo a los compradores de los no compradores? Traza una línea recta (un hiperplano) en el espacio 2D. 
A un lado de la línea, la probabilidad calculada es $< 0.5$ (predice 0). Al otro lado, es $\geq 0.5$ (predice 1).

In [ ]:
from matplotlib.colors import ListedColormap

# Función auxiliar para visualizar la frontera
def graficar_frontera(X_set, y_set, modelo, title):
    X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                         np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
    plt.figure(figsize=(8, 6))
    plt.contourf(X1, X2, modelo.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
                 alpha = 0.3, cmap = ListedColormap(('red', 'green')))
    plt.xlim(X1.min(), X1.max())
    plt.ylim(X2.min(), X2.max())
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                    c = [ListedColormap(('red', 'green'))(i)], label = j, edgecolor='black')
    plt.title(title)
    plt.xlabel('Edad (Escalada)')
    plt.ylabel('Salario (Escalado)')
    plt.legend()
    plt.show()

graficar_frontera(X_train_scaled, y_train, modelo_logistico, 'Frontera de Decisión (Datos de Entrenamiento)')

La línea recta que divide las zonas roja (predicción=0) y verde (predicción=1) es la frontera matemática que el algoritmo ha aprendido.

## 4. Evaluación del Modelo de Clasificación

A diferencia de la regresión lineal donde usábamos Error Cuadrático Medio (MSE), en clasificación contamos aciertos y errores en 4 categorías:
- **Verdaderos Positivos (VP):** El modelo predijo que compró (1) y el usuario REALMENTE compró (1).
- **Verdaderos Negativos (VN):** El modelo predijo que NO compró (0) y REALMENTE NO compró (0).
- **Falsos Positivos (FP):** El modelo predijo que compró (1), pero en realidad NO compró (0).
- **Falsos Negativos (FN):** El modelo predijo que NO compró (0), pero en realidad SÍ compró (1).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Generamos predicciones sobre el conjunto de prueba
y_pred = modelo_logistico.predict(X_test_scaled)

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicción: No Compró (0)', 'Predicción: Compró (1)'],
            yticklabels=['Real: No Compró (0)', 'Real: Compró (1)'])
plt.title('Matriz de Confusión')
plt.show()

# Reporte de Clasificación (Precision, Recall, F1-Score)
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

### Conclusión de Negocio

Interpretar la **Precisión** y el **Recall** es fundamental:
- **Precisión (de los que predije positivos, ¿cuántos lo son realmente?):** Si lanzamos una campaña costosa de marketing a los que predijimos como compradores, una alta precisión significa que no estamos desperdiciando dinero en falsos positivos.
- **Recall (de todos los positivos reales, ¿cuántos logré atrapar?):** Si no queremos dejar escapar a NINGÚN cliente potencial, buscamos un recall alto, intentando reducir los falsos negativos.

¡Felicidades! Has entrenado tu primer modelo de clasificación.